# Transformer(2.Multi-Head Attention) 언어 모델 총정리 

## 1. Multi-Head Attention 공식
Self-Attention을 Head 개수 만큼 계산

---
### 1) Query, Key, Value 계산(생성) : 
### $ [ Q = XW_Q, \quad K = XW_K, \quad V = XW_V  ] $
### 설명)
입력벡터 $ [ X = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} ] $

가중치 행렬 곱으로 Q, K, V 생성 :

Query: $ [ Q = XW_Q ] $

Key: $ [ K = XW_K ] $

Value: $ [ V = XW_V ] $

여기서 $ [ W_Q, W_K, W_V ] $ 는 학습되는 파라미터입니다.

	◦ Query는 “내가 다른 토큰에게 무엇을 물어보고 싶은가?”
	◦ Key는 “내가 가진 정보는 무엇인가?”
	◦ Value는 “내가 실제로 전달할 정보는 무엇인가?”
    
---
### 2) 유사도(Score) 계산 : 
### $ [ \text{Score} = QK^\top ] $
### 설명)
Query와 Key를 내적(dot product)해서 유사도를 구합니다:
$ [ \text{Score} = QK^\top ] $

	◦ Query는 “질문자”
	◦ Key는 “답변자”
	◦ 내적 결과는 “질문자와 답변자가 얼마나 잘 맞는가?”를 수치화한 것
	◦ 즉, Score는 **토큰 간 관계(연관성)**를 나타냅니다.

---
### 3) 스케일링(Scaling) : 
### $ [ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} ] $
### 설명)
- 스케일링
$ [ \text{Scaled Score} = \frac{QK^\top}{\sqrt{d_k}} ] $
→ 값이 너무 커지지 않도록 Key 차원수로 나눠 안정화합니다.
---
### 4) Softmax 변환 : 
### $ [ \text{Attention Weights} = \text{Softmax}(\text{Scaled Score}) ] $
### 설명)
- Softmax 변환
$ [ \text{Attention Weights} = \text{Softmax}(\text{Scaled Score}) ] $ 
→ 각 토큰이 다른 토큰을 얼마나 참고해야 하는지 확률 분포로 변환합니다.
---
### 5) Value 가중합 (최종 출력) : $ [ \text{Output} = \text{Attention Weights} \cdot V ] $
### 설명)
- Attention Weights를 Value에 곱해 최종 출력을 얻습니다 : 
$ [ \text{Output} = \text{Attention Weights} \cdot V ] $

---
### 6) Concat 단계 :
### 설명)
- 두 헤드를 이어 붙인다 : 
$ [ Z = \text{Concat}(\text{Output}_1, \text{Output}_2) 
= 예시로 \begin{bmatrix}0.67 & 0.33 & 0.67 & 0.33 \\ 0.5 & 0.5 & 0.5 & 0.5\end{bmatrix}
 ] $
 
---
### 7) 출력 가중치 $ [W_O] $ 곱 계산 : 
### 설명)
- Concat 출력값에 출력가중치 $ [W_O] $ 곱 계산
- 예시로 $ [ W_O ]를 4×2 $ 행렬로 설정(초기에 가중치 랜덤값으로 설정):
$ [ W_O = \begin{bmatrix}1 & 0 \\ 0 & 1 \\ 1 & 0 \\ 0 & 1\end{bmatrix} ] $

최종 출력:
$ [ \text{MultiHead}(X) = Z W_O ] $

계산:

• 첫 번째 행: $ [ [0.67, 0.33, 0.67, 0.33] \cdot W_O = [1.34, 0.66] ] $

• 두 번째 행: $ [ [0.5, 0.5, 0.5, 0.5] \cdot W_O = [1.0, 1.0] ] $

따라서:
$ [ \text{MultiHead}(X) = \begin{bmatrix}1.34 & 0.66 \\ 1.0 & 1.0\end{bmatrix} ] $

	◦ 헤드별 Query/Key/Value 생성 → 헤드별 Query와 Key로 유사도 계산 → 헤드별 스케일링 계산 → 헤드별 Softmax로 확률화 적용 → 헤드별 Value를 가중합 
	→ Concat(Head 들을 이어 붙인다) → 출력가중치 $ [W_O] $ 곱 계산 → 하는 것이 Multi-Head Attention의 핵심입니다.

---

## 2. Transformer(2.Multi-Head Attention) 언어 모델의 Multi-Head Attention 계산 과정을 공식과 함께 단계별로 계산
---
### 1) 입력과 초기 가중치
### 1-1) 입력 벡터:
$ [ X = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} ] $
(두 개의 토큰, 각 토큰은 2차원)

### 1-2) Head 1 가중치:
$ [ W_Q^{(1)} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix}, \quad W_K^{(1)} = \begin{bmatrix}1 & 1 \\ 0 & 1\end{bmatrix}, \quad W_V^{(1)} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix} ] $

### 1-3) Head 2 가중치:
$ [ W_Q^{(2)} = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}, \quad W_K^{(2)} = \begin{bmatrix}1 & 0 \\ 1 & 1\end{bmatrix}, \quad W_V^{(2)} = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}
 ] $
 
---
### 2) Head 1 계산
### 2-1) Query, Key, Value 계산(생성) :
$ [ Q_1 = XW_Q^{(1)} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix}, \quad
K_1 = XW_K^{(1)} = \begin{bmatrix}1 & 1 \\ 0 & 1\end{bmatrix}, \quad
V_1 = XW_V^{(1)} = \begin{bmatrix}1 & 0 \\ 0 & 1\end{bmatrix}
 ] $

### 2-2) Score 유사도 계산 :
$ [ Q_1K_1^\top = \begin{bmatrix}1 & 0 \\ 1 & 1\end{bmatrix} ] $

### 2-3) 스케일링 계산 $ ([ \sqrt{d_k}=1.414 ]) $ :
$ [ \begin{bmatrix}0.71 & 0 \\ 0.71 & 0.71\end{bmatrix} ] $

### 2-4) Softmax 적용 :
$ [ \text{Attention Weights}_1 = \begin{bmatrix}0.67 & 0.33 \\ 0.5 & 0.5\end{bmatrix} ] $

### 2-5) Value 가중합 :
$ [ \text{Output}_1 = \text{Attention Weights}_1 V_1 = \begin{bmatrix}0.67 & 0.33 \\ 0.5 & 0.5\end{bmatrix} ] $

---
### 3) Head 2 계산
### 3-1) Query, Key, Value 계산(생성) :
$ [ Q_2 = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}, \quad
K_2 = \begin{bmatrix}1 & 0 \\ 1 & 1\end{bmatrix}, \quad
V_2 = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}
 ] $

### 3-2) Score 유사도 계산 :
$ [ Q_2K_2^\top = \begin{bmatrix}0 & 1 \\ 1 & 1\end{bmatrix} ] $

### 3-3) 스케일링 계산 $ ([ \sqrt{d_k}=1.414 ]) $ :
$ [ \begin{bmatrix}0 & 0.71 \\ 0.71 & 0.71\end{bmatrix} ] $

### 3-4) Softmax 적용 :
$ [ \text{Attention Weights}_2 = \begin{bmatrix}0.33 & 0.67 \\ 0.5 & 0.5\end{bmatrix} ] $

### 3-5) Value 가중합 :
$ [ \text{Output}_2 = \text{Attention Weights}_2 V_2 = \begin{bmatrix}0.67 & 0.33 \\ 0.5 & 0.5\end{bmatrix} ] $

---
### 4) Concat 단계 :
$ [ Z = \text{Concat}(\text{Output}_1, \text{Output}_2) =
\begin{bmatrix}0.67 & 0.33 & 0.67 & 0.33 \\ 0.5 & 0.5 & 0.5 & 0.5\end{bmatrix}
 ] $
 
---
### 5) 출력 가중치 $ [W_O] $ 곱 계산 : 
예시로 $ [ W_O ]를 4×2 $ 행렬로 설정:
$ [ W_O = \begin{bmatrix}1 & 0 \\ 0 & 1 \\ 1 & 0 \\ 0 & 1\end{bmatrix} ] $

최종 출력:
$ [ \text{MultiHead}(X) = Z W_O ] $

계산:

• 첫 번째 행: $ [ [0.67, 0.33, 0.67, 0.33] \cdot W_O = [1.34, 0.66] ] $

• 두 번째 행: $ [ [0.5, 0.5, 0.5, 0.5] \cdot W_O = [1.0, 1.0] ] $

따라서:
$ [ \text{MultiHead}(X) = \begin{bmatrix}1.34 & 0.66 \\ 1.0 & 1.0\end{bmatrix} ] $

---
### 6) 최종 정리
	◦ 멀티헤드 어텐션은 각 헤드별 Attention → Concat → [ W_O ] 선형 변환으로 완성됩니다.
	◦ Concat은 중간 단계이고, 반드시 [ W_O ]를 곱해 최종 출력 차원으로 맞춰야 합니다.
	◦ 이 최종 출력이 트랜스포머 인코더 블록의 다음 단계(잔차 연결 + LayerNorm + FFN)로 전달됩니다.